# Generate options book JSON

Build `options_book.json` from parameter grids. Maturities use `expiry_years` (year fraction from market `asof`).


1. Edit the grids in the next cell (tenors, strikes, product toggles).
2. Run all cells → writes `OUTPUT_JSON`.
3. Price with [`pricing_from_json.ipynb`](pricing_from_json.ipynb).

Products supported: `european`, `digital`, `digital_accrual`, `asian`, `barrier`, `lookback`, `autocall`.

In [23]:
import json
from itertools import product
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent.parent
elif not (REPO / "CMakeLists.txt").exists():
    for p in REPO.parents:
        if (p / "CMakeLists.txt").exists():
            REPO = p
            break

OUTPUT_JSON = REPO / "data" / "options_book.json"

# --- maturities (year fractions from market asof) ---
# Per-product override; omit a product to use DEFAULT_EXPIRY_YEARS.
DEFAULT_EXPIRY_YEARS = [0.5, 1.0, 1.5, 2.0]
EXPIRY_YEARS_BY_PRODUCT = {
    "barrier": [0.5, 1.5],  # 1.5 used only for daily-monitoring extras
    "autocall": [0.5, 1.0, 1.5],
    "lookback": [0.5, 1.0, 1.5, 2.0],
    "digital_accrual": [1.0, 2.0],
    "asian": [ 1.0, 2.0],
}
BARRIER_RECIPE_TENORS = [0.5]  # full barrier grid on these tenors only

STRIKES = [0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15]

# --- product toggles ---
INCLUDE_EUROPEAN = False
INCLUDE_DIGITAL = True
INCLUDE_DIGITAL_ACCRUAL = False  
INCLUDE_ASIAN = True
INCLUDE_BARRIER = True
INCLUDE_LOOKBACK = True       
INCLUDE_AUTOCALL = True

# --- product-specific grids ---
EUROPEAN = {"is_call": [True, False], "quote_space": ["S"]}
DIGITAL = {"is_call": [True], "quote_space": ["S"], "asset_or_nothing": [False, True]}
DIGITAL_ACCRUAL = {
    "strike_low_fraction_of_spot": [0.90],
    "strike_up_fraction_of_spot": [1.10],
    "quote_space": ["S"],
    "observation_frequency": ["monthly"],  # or "daily"
}
ASIAN = {
    "average_type": ["arithmetic", "geometric"],
    "strike_style": ["fixed", "floating"],
    "is_call": [True],
    "strikes": STRIKES,
    "observation_frequency": [None],  # monthly default; add "daily" for extra rows
}
# Extra daily-monitoring examples (tenor -> list of recipe dicts)
ASIAN_DAILY_EXTRAS = {
    1.5: [{"strike": 1.00, "average_type": "arithmetic", "strike_style": "fixed"}],
}
BARRIER_RECIPES = [
    {"barrier_type": "up_out", "strike": 0.80, "barrier_up": b}
    for b in [1.05, 1.10, 1.15, 1.20]
] + [
    {"barrier_type": "up_in", "strike": 0.80, "barrier_up": b}
    for b in [1.05, 1.10, 1.15, 1.20]
] + [
    {"barrier_type": "down_out", "strike": 0.85, "barrier_down": 0.80},
    {"barrier_type": "down_in", "strike": 0.85, "barrier_down": 0.80},
] + [
    {"barrier_type": "up_out", "strike": s, "barrier_up": b}
    for s in [0.85, 0.90, 0.95, 1.00]
    for b in [1.05, 1.10, 1.15, 1.20]
] + [
    {"barrier_type": "up_in", "strike": s, "barrier_up": b}
    for s in [0.85, 0.90, 0.95, 1.00]
    for b in [1.05, 1.10, 1.15, 1.20]
]
BARRIER_DAILY_EXTRAS = {
    1.5: [{"barrier_type": "down_out", "strike": 1.00, "barrier_down": 0.85}],
}
LOOKBACK = {
    "strike_style": ["fixed", "floating"],
    "is_call": [True],
    "strikes": [0.80, 0.90, 1.0, 1.10, 1.20],  
    "observation_frequency": [None],
}
AUTOCALL = {
    "coupon_style": ["phoenix", "athena"],
    "coupon_rate_per_period": [0.02, 0.03, 0.04],
    "barrier_fraction_of_spot": [0.90, 1.00, 1.10],
}


def expiries_for(product: str) -> list[float]:
    return EXPIRY_YEARS_BY_PRODUCT.get(product, DEFAULT_EXPIRY_YEARS)

In [24]:
def t_label(years: float) -> str:
    if years == int(years):
        return f"{int(years)}y"
    return f"{years:g}y"


def k_label(k: float) -> str:
    return f"{k:.2f}"


def row(**fields):
    base = {"id", "product", "expiry_years"}
    out = {k: fields[k] for k in ("id", "product", "expiry_years")}
    for k in sorted(fields):
        if k in base or fields[k] is None:
            continue
        out[k] = fields[k]
    return out


def cartesian_grid(fixed: dict, grid: dict):
    keys = list(grid.keys())
    for values in product(*(grid[k] for k in keys)):
        item = dict(fixed)
        for k, v in zip(keys, values):
            item[k] = v
        yield item


def build_european():
    out = []
    for ty in expiries_for("european"):
        for strike in STRIKES:
            for cfg in cartesian_grid({}, EUROPEAN):
                side = "call" if cfg["is_call"] else "put"
                oid = f"european_{side}_{k_label(strike)}_{t_label(ty)}"
                out.append(row(
                    id=oid, product="european", expiry_years=ty,
                    strike_fraction_of_spot=strike, **cfg,
                ))
    return out


def build_digital():
    out = []
    for ty in expiries_for("digital"):
        for strike in STRIKES:
            for cfg in cartesian_grid({}, DIGITAL):
                kind = "asset" if cfg["asset_or_nothing"] else "cash"
                oid = f"digital_{kind}_{k_label(strike)}_{t_label(ty)}"
                out.append(row(
                    id=oid, product="digital", expiry_years=ty,
                    strike_fraction_of_spot=strike, **cfg,
                ))
    return out


def build_digital_accrual():
    out = []
    for ty in expiries_for("digital_accrual"):
        for cfg in cartesian_grid({}, DIGITAL_ACCRUAL):
            lo, up = cfg["strike_low_fraction_of_spot"], cfg["strike_up_fraction_of_spot"]
            freq = cfg.get("observation_frequency", "monthly")
            oid = f"digital_accrual_{k_label(lo)}_{k_label(up)}_{freq}_{t_label(ty)}"
            out.append(row(
                id=oid, product="digital_accrual", expiry_years=ty,
                strike_low_fraction_of_spot=lo,
                strike_up_fraction_of_spot=up,
                quote_space=cfg["quote_space"],
                observation_frequency=freq,
            ))
    return out


def _append_asian_row(out, ty, strike, cfg, freq=None):
    oid = (
        f"asian_{cfg['average_type']}_{cfg['strike_style']}_{k_label(strike)}"
        + (f"_{freq}" if freq else "")
        + f"_{t_label(ty)}"
    )
    fields = dict(cfg, strike_fraction_of_spot=strike)
    if freq:
        fields["observation_frequency"] = freq
    out.append(row(id=oid, product="asian", expiry_years=ty, **fields))


def build_asian():
    out = []
    grid = {k: v for k, v in ASIAN.items() if k != "strikes"}
    for ty in expiries_for("asian"):
        for strike in ASIAN["strikes"]:
            for cfg in cartesian_grid({}, grid):
                cfg = dict(cfg)
                freq = cfg.pop("observation_frequency")
                _append_asian_row(out, ty, strike, cfg, freq)
        for extra in ASIAN_DAILY_EXTRAS.get(ty, []):
            extra = dict(extra)
            strike = extra.pop("strike")
            _append_asian_row(out, ty, strike, extra, "daily")
    return out


def _append_barrier_row(out, ty, rc):
    rc = dict(rc)
    freq = rc.pop("observation_frequency", None)
    btype = rc["barrier_type"]
    strike = rc["strike"]
    parts = [f"barrier_{btype}", k_label(strike)]
    fields = {
        "barrier_type": btype,
        "is_call": True,
        "strike_fraction_of_spot": strike,
    }
    if "barrier_up" in rc:
        fields["barrier_up_fraction_of_spot"] = rc["barrier_up"]
        parts.append(k_label(rc["barrier_up"]))
    if "barrier_down" in rc:
        fields["barrier_down_fraction_of_spot"] = rc["barrier_down"]
        parts.append(k_label(rc["barrier_down"]))
    if freq:
        fields["observation_frequency"] = freq
        parts.append(freq)
    parts.append(t_label(ty))
    out.append(row(id="_".join(parts), product="barrier", expiry_years=ty, **fields))


def build_barrier():
    out = []
    for ty in expiries_for("barrier"):
        if ty in BARRIER_RECIPE_TENORS:
            for rc in BARRIER_RECIPES:
                _append_barrier_row(out, ty, rc)
        for rc in BARRIER_DAILY_EXTRAS.get(ty, []):
            _append_barrier_row(out, ty, {**rc, "observation_frequency": "daily"})
    return out


def build_lookback():
    out = []
    grid = {k: v for k, v in LOOKBACK.items() if k != "strikes"}
    for ty in expiries_for("lookback"):
        for cfg in cartesian_grid({}, grid):
            cfg = dict(cfg)
            freq = cfg.pop("observation_frequency")
            style = cfg["strike_style"]
            side = "call" if cfg["is_call"] else "put"
            if style == "floating":
                # strike ignored; one row per (tenor, style, call/put)
                if not cfg["is_call"]:
                    continue  # floating put on max(S) is identically zero — see schema
                oid = f"lookback_{side}_{style}_{t_label(ty)}"
                fields = dict(cfg)
            else:
                for strike in LOOKBACK["strikes"]:
                    oid = f"lookback_{side}_{style}_{k_label(strike)}_{t_label(ty)}"
                    fields = dict(cfg, strike_fraction_of_spot=strike)
                    if freq:
                        fields["observation_frequency"] = freq
                    out.append(row(id=oid, product="lookback", expiry_years=ty, **fields))
                continue
            if freq:
                fields["observation_frequency"] = freq
            out.append(row(id=oid, product="lookback", expiry_years=ty, **fields))
    return out


def build_autocall():
    out = []
    for ty in expiries_for("autocall"):
        for cfg in cartesian_grid({}, AUTOCALL):
            oid = (
                f"autocall_{cfg['coupon_style']}"
                f"_b{k_label(cfg['barrier_fraction_of_spot'])}"
                f"_c{k_label(cfg['coupon_rate_per_period'])}"
                f"_{t_label(ty)}"
            )
            out.append(row(id=oid, product="autocall", expiry_years=ty, **cfg))
    return out


BUILDERS = [
    (INCLUDE_EUROPEAN, build_european),
    (INCLUDE_DIGITAL, build_digital),
    (INCLUDE_DIGITAL_ACCRUAL, build_digital_accrual),
    (INCLUDE_ASIAN, build_asian),
    (INCLUDE_BARRIER, build_barrier),
    (INCLUDE_LOOKBACK, build_lookback),
    (INCLUDE_AUTOCALL, build_autocall),
]


def build_options_book():
    options = []
    for enabled, fn in BUILDERS:
        if enabled:
            options.extend(fn())
    ids = [o["id"] for o in options]
    if len(ids) != len(set(ids)):
        from collections import Counter
        dupes = [k for k, v in Counter(ids).items() if v > 1]
        raise ValueError(f"duplicate ids: {dupes[:5]}")
    return options

In [25]:
options = build_options_book()
payload = {"options": options}

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(payload, indent=2) + "\n")

from collections import Counter
by_product = Counter(o["product"] for o in options)
by_tenor = Counter(o["expiry_years"] for o in options)

print(f"wrote {len(options)} options → {OUTPUT_JSON}")
print("by product:", dict(sorted(by_product.items())))
print("by expiry_years:", dict(sorted(by_tenor.items())))

wrote 249 options → /Users/martino/Desktop/Pricing_Engine_QuantLib/data/options_book.json
by product: {'asian': 64, 'autocall': 54, 'barrier': 43, 'digital': 64, 'lookback': 24}
by expiry_years: {0.5: 82, 1.0: 72, 1.5: 41, 2.0: 54}


In [26]:
# preview first 3 rows per product
seen = set()
for o in options:
    if o["product"] in seen:
        continue
    seen.add(o["product"])
    print(json.dumps(o, indent=2))
    print()

{
  "id": "digital_cash_0p80_0p5y",
  "product": "digital",
  "expiry_years": 0.5,
  "asset_or_nothing": false,
  "is_call": true,
  "quote_space": "S",
  "strike_fraction_of_spot": 0.8
}

{
  "id": "asian_arithmetic_fixed_0p80_1y",
  "product": "asian",
  "expiry_years": 1.0,
  "average_type": "arithmetic",
  "is_call": true,
  "strike_fraction_of_spot": 0.8,
  "strike_style": "fixed"
}

{
  "id": "barrier_up_out_0p80_1p05_0p5y",
  "product": "barrier",
  "expiry_years": 0.5,
  "barrier_type": "up_out",
  "barrier_up_fraction_of_spot": 1.05,
  "is_call": true,
  "strike_fraction_of_spot": 0.8
}

{
  "id": "lookback_call_fixed_0p80_0p5y",
  "product": "lookback",
  "expiry_years": 0.5,
  "is_call": true,
  "strike_fraction_of_spot": 0.8,
  "strike_style": "fixed"
}

{
  "id": "autocall_phoenix_b0p90_c0p02_0p5y",
  "product": "autocall",
  "expiry_years": 0.5,
  "barrier_fraction_of_spot": 0.9,
  "coupon_rate_per_period": 0.02,
  "coupon_style": "phoenix"
}

